# Remote SSH Validation Notebook for Adeept RaspTank Pro

This notebook is designed to be branch-aware and safe.

Important:
- Some wrappers exist on the refactor branch, but not every script/class has been implemented yet.
- This notebook checks what is actually present before importing or testing it.
- Do not run real motion until the robot is physically secured and the test is intentionally tiny.


In [ ]:
# Cell 1: Find the repo on the Raspberry Pi and confirm the path.
from pathlib import Path
import os

possible_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.home() / 'Adeept_RaspTankPro',
    Path('/home/pi/Adeept_RaspTankPro'),
    Path('/workspace'),
]

repo_root = None
for p in possible_roots:
    if p.exists() and (p / 'rasptank').exists():
        repo_root = p
        break

print('CWD =', Path.cwd())
print('HOME =', Path.home())
if repo_root is None:
    print('Repo root not found. Open the repository folder in VS Code Remote SSH before continuing.')
else:
    print('Repo root found:', repo_root)


In [ ]:
# Cell 2: Add the repo to sys.path only if it was found.
import sys
from pathlib import Path

repo_root = None
for candidate in [
    Path.cwd(),
    Path.cwd().parent,
    Path.home() / 'Adeept_RaspTankPro',
    Path('/home/pi/Adeept_RaspTankPro'),
    Path('/workspace'),
]:
    if candidate.exists() and (candidate / 'rasptank').exists():
        repo_root = candidate
        break

if repo_root is not None:
    sys.path.insert(0, str(repo_root))
    print('Inserted repo root:', repo_root)
else:
    print('Repository not found on this machine. Please open the repo folder first.')


In [ ]:
# Cell 3: Check whether the expected package modules exist in the current branch.
# This avoids failing immediately when the branch is only partially refactored.
import importlib.util

required_modules = [
    'rasptank',
    'rasptank.config',
    'rasptank.tank',
    'rasptank.hardware',
    'rasptank.hardware.motors',
    'rasptank.hardware.servos',
    'rasptank.hardware.ultrasonic',
    'rasptank.hardware.leds',
    'rasptank.hardware.display',
]

for name in required_modules:
    spec = importlib.util.find_spec(name)
    status = 'FOUND' if spec is not None else 'MISSING'
    print(f'{name}: {status}')


In [ ]:
# Cell 4: Import only the modules that are actually available in this branch.
# This prevents a notebook crash when a class or module is still under development.

import importlib

import_status = {}
for name in [
    'rasptank',
    'rasptank.hardware.motors',
    'rasptank.hardware.servos',
    'rasptank.hardware.ultrasonic',
    'rasptank.hardware.leds',
    'rasptank.hardware.display',
    'rasptank.tank',
]:
    try:
        importlib.import_module(name)
        import_status[name] = 'OK'
    except Exception as exc:
        import_status[name] = f'NOT_AVAILABLE: {type(exc).__name__}: {exc}'

for name, status in import_status.items():
    print(f'{name}: {status}')


In [ ]:
# Cell 5: Safe validation of the robot API.
# If a class is missing, this prints a message instead of crashing the notebook.

try:
    from rasptank import RaspTank
    print('RaspTank imported successfully')
    tank = RaspTank()
    print('RaspTank instantiated:', tank)
except Exception as exc:
    print('RaspTank unavailable:', type(exc).__name__, exc)


## Safety gate before real movement

Do not run the motion cells until all of the following are true:
- the repo is open on the Raspberry Pi
- the correct remote kernel is selected
- imports succeeded for the classes you intend to test
- the robot is physically secured on a stand or table
- the speed is low
- the test lasts only a fraction of a second


In [ ]:
# Cell 6: Short low-power motion test
# This remains disabled by default to protect the robot and the user.
import time

RUN_SAFE_TEST = False

if RUN_SAFE_TEST:
    try:
        from rasptank import RaspTank
        tank = RaspTank()
        print('Starting tiny forward motion test...')
        tank.forward(15)
        time.sleep(0.3)
        tank.stop()
        print('Test complete. Robot stopped.')
    except Exception as exc:
        print('Motion test failed to start:', type(exc).__name__, exc)
    finally:
        try:
            tank.close()
        except Exception:
            pass
else:
    print('SAFE_TEST_DISABLED: set RUN_SAFE_TEST = True only when the robot is safely secured and the test is intentionally tiny.')
